# Option 4: Synthetic Control Method (SCM)

**Replication**: Abadie, Diamond & Hainmueller (2010) — Synthetic Control Methods for Comparative Case Studies

**Key Question**: Did California's Proposition 99 (a tobacco tax and anti-smoking program enacted in 1988) reduce per-capita cigarette sales?

We use the original dataset from the paper: 39 US states, 1970–2000, with California as the treated unit and 38 states as potential donors.

## 1. Setup and Data Loading

The data comes from Scott Cunningham's Causal Inference: The Mixtape. It is bundled locally as a CSV file.

In [ ]:
# Download data files if not already present
import os
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/JasmineHao/JasmineHao.github.io/main/econ6083/final-project/notebooks/data/"
DATA_FILES = ['proposition99.csv', 'china_city_panel_with_policies.csv']

os.makedirs('data', exist_ok=True)
for fname in DATA_FILES:
    if not os.path.exists(f'data/{fname}'):
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(BASE_URL + fname, f'data/{fname}')
        print(f"  Saved to data/{fname}")
    else:
        print(f"Found local: data/{fname}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize



df = pd.read_csv('data/proposition99.csv')
print("Loaded Proposition 99 data from local file")

print(f"Dataset shape: {df.shape}")
print(f"States: {df['state'].nunique()}")
print(f"Years: {int(df['year'].min())}-{int(df['year'].max())}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Data Exploration

Identify California and create the treatment indicator.

In [ ]:
df['treated'] = (df['state'] == 'California').astype(int)
df['post'] = (df['year'] >= 1988).astype(int)
TREAT_YEAR = 1988

ca = df[df['state'] == 'California'][['year', 'cigsale']]
print(ca.head())

avg = df.groupby(['year', 'treated'])['cigsale'].mean().reset_index()
print(avg.pivot(index='year', columns='treated', values='cigsale').head())

## 3. Visualise Pre-Treatment Trajectories

Plot California alongside all donor states in the pre-treatment period.

In [ ]:
pre = df[df['year'] < TREAT_YEAR]['year'].unique()
fig, ax = plt.subplots(figsize=(10, 5))
for s in df['state'].unique():
    if s != 'California':
        sd = df[df['state'] == s]
        ax.plot(sd['year'], sd['cigsale'], color='lightgray', alpha=0.5, linewidth=0.8)
ca_df = df[df['state'] == 'California']
ax.plot(ca_df['year'], ca_df['cigsale'], color='steelblue', linewidth=2.5, label='California')
ax.axvline(TREAT_YEAR - 0.5, color='red', linestyle='--', label='Proposition 99')
ax.set_xlabel('Year')
ax.set_ylabel('Cigarette sales (packs per 100k)')
ax.set_title('Cigarette Sales: California vs Donor States')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Build the Synthetic Control

Choose donor weights to minimise pre-treatment MSE.

In [ ]:
wide = df.pivot(index='year', columns='state', values='cigsale')
y_ca = wide['California'].values
donors = wide.drop(columns='California')
names = donors.columns.tolist()
X = donors.values
n_pre = sum(wide.index < TREAT_YEAR)
y_pre = y_ca[:n_pre]
X_pre = X[:n_pre, :]

def obj(w):
    return np.mean((y_pre - X_pre @ w) ** 2)

res = minimize(obj, np.ones(len(names))/len(names),
               method='SLSQP', bounds=[(0,1)]*len(names),
               constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
w_opt = res.x
syn = X @ w_opt

w_df = pd.DataFrame({'state': names, 'weight': w_opt})
w_df = w_df[w_df['weight'] > 0.001].sort_values('weight', ascending=False)
print(w_df.to_string(index=False))
r2 = 1 - np.var(y_pre - X_pre @ w_opt) / np.var(y_pre)
print(f"Pre-treatment R2: {r2:.4f}")

## 5. Visualise California vs Synthetic Control

In [ ]:
years = wide.index.values
gap = y_ca - syn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.plot(years, y_ca, 'o-', color='steelblue', label='California', linewidth=2)
ax.plot(years, syn, 's--', color='coral', label='Synthetic', linewidth=2)
ax.axvline(TREAT_YEAR - 0.5, color='red', linestyle='--', label='Proposition 99')
ax.set_xlabel('Year')
ax.set_ylabel('Cigarette sales')
ax.set_title('California vs Synthetic Control')
ax.legend()
ax = axes[1]
ax.plot(years, gap, 'o-', color='darkgreen', linewidth=2)
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.axvline(TREAT_YEAR - 0.5, color='red', linestyle='--')
ax.fill_between(years, gap, 0, where=(years >= TREAT_YEAR), alpha=0.2, color='green')
ax.set_xlabel('Year')
ax.set_ylabel('Gap')
ax.set_title('Estimated Treatment Effect')
plt.tight_layout()
plt.show()
print(f"Post-treatment avg gap: {gap[n_pre:].mean():.2f}")
print(f"Gap in 2000: {gap[-1]:.2f}")

## 6. Placebo Test (In-Space Placebo)

In [ ]:
placebo_gaps = []
for donor in names:
    y_p = wide[donor].values
    pool = wide.drop(columns=['California', donor])
    Xp = pool.values
    yp = y_p[:n_pre]
    Xp_pre = Xp[:n_pre, :]
    def op(w): return np.mean((yp - Xp_pre @ w)**2)
    rp = minimize(op, np.ones(Xp_pre.shape[1])/Xp_pre.shape[1],
                  method='SLSQP', bounds=[(0,1)]*Xp_pre.shape[1],
                  constraints={'type':'eq','fun':lambda w: np.sum(w)-1})
    syn_p = Xp @ rp.x
    placebo_gaps.append(y_p - syn_p)

fig, ax = plt.subplots(figsize=(10, 5))
for pg in placebo_gaps:
    ax.plot(years, pg, color='lightgray', alpha=0.5, linewidth=0.7)
ax.plot(years, gap, color='steelblue', linewidth=2.5, label='California')
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.axvline(TREAT_YEAR - 0.5, color='red', linestyle='--', label='Proposition 99')
ax.set_xlabel('Year')
ax.set_ylabel('Gap')
ax.set_title('In-Space Placebo Test')
ax.legend()
plt.tight_layout()
plt.show()

post_rmspe_ca = np.sqrt(np.mean(gap[n_pre:]**2))
post_rmspe_p = [np.sqrt(np.mean(pg[n_pre:]**2)) for pg in placebo_gaps]
print(f"CA post-RMSPE: {post_rmspe_ca:.3f}")
print(f"Median placebo post-RMSPE: {np.median(post_rmspe_p):.3f}")
print(f"RMSPE ratio: {post_rmspe_ca/np.median(post_rmspe_p):.2f}")
n_ex = sum(1 for p in post_rmspe_p if post_rmspe_ca > p)
print(f"CA gap exceeds {n_ex}/{len(post_rmspe_p)} placebos")

## Interpreting Your Results

| Output | What it means | What to look for |
|---|---|---|
| **Pre-treatment R²** | How well synthetic CA matches actual CA pre-1988 | > 0.95 is excellent; < 0.80 raises concerns |
| **Donor weights** | Which states contribute to the synthetic control | Do they make sense geographically/economically? |
| **Post-treatment gap** | Estimated effect = Actual CA − Synthetic CA | Negative = sales fell due to Proposition 99 |
| **Placebo RMSPE ratio** | How unusual is CA's gap relative to donors | Ratio > 2 suggests a genuine effect |

**Key question**: Would California's cigarette sales have continued along the synthetic control trajectory without Proposition 99? The gap after 1988 is your causal estimate.

## Summary

| Finding | Value | Interpretation |
|---|---|---|
| Pre-treatment fit (R2) | ~0.97 | Synthetic California closely tracks actual California before 1988 |
| Post-treatment gap (2000) | ~-25 to -30 packs | Cigarette sales fell by ~25-30 packs per 100k people |
| Placebo test | CA > most donors | The divergence is unusually large |
| Top donors | Colorado, Connecticut, Montana, Nevada, Utah | States weighted most heavily |

**Takeaway**: The synthetic control shows that California's cigarette sales would have followed a trajectory similar to the weighted combination of donor states had Proposition 99 not been enacted. The sharp divergence after 1988 suggests the policy reduced cigarette consumption. This matches the original Abadie et al. (2010) finding.